In [54]:
import numpy as np
import glob
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
)
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

# --- Load Embeddings and Labels ---
save_dir = '/Users/muhammad/Desktop/CIDDS'

embedding_files = sorted(glob.glob(f"{save_dir}/embeddings_batch_*.npy"))
label_files = sorted(glob.glob(f"{save_dir}/labels_batch_*.npy"))




In [55]:
label_files

['/Users/muhammad/Desktop/CIDDS/labels_batch_0.npy',
 '/Users/muhammad/Desktop/CIDDS/labels_batch_1.npy']

In [52]:
np = np.load(label_files[0])

In [53]:
np

array(['Benign', 'Benign', 'Benign', ..., 'Benign', 'Benign', 'Benign'],
      dtype='<U14')

In [59]:
X = np.concatenate([np.load(f) for f in embedding_files])

y = np.concatenate([np.load(f) for f in label_files])


In [60]:
X

array([[[ 0.13574435, -0.05160529, -0.50275964, ...,  0.67074317,
          1.1534138 ,  1.1073054 ],
        [-0.22965957, -0.8483704 , -0.22614595, ...,  0.47493333,
          0.5455789 ,  1.199609  ],
        [ 0.13577531, -0.0516167 , -0.50274026, ...,  0.67073464,
          1.1533757 ,  1.1074128 ],
        ...,
        [ 0.1357471 , -0.05163427, -0.5027634 , ...,  0.6707652 ,
          1.1534197 ,  1.1073087 ],
        [-0.06740427, -0.87540287, -0.138461  , ...,  1.5017185 ,
          2.5422926 ,  0.7547021 ],
        [ 0.13578233, -0.0516123 , -0.5027347 , ...,  0.67072606,
          1.1533653 ,  1.1074396 ]],

       [[ 0.01673471, -0.9487314 , -0.23607543, ...,  0.692721  ,
          0.41137505,  1.615559  ],
        [-0.01555301, -0.5426343 ,  0.0241118 , ...,  0.9655171 ,
          0.6905954 ,  1.1050287 ],
        [-0.19944161, -0.7471315 , -0.20164286, ...,  1.2202778 ,
          1.5904628 ,  1.0526913 ],
        ...,
        [ 0.13619977, -1.0130551 , -0.29435995, ...,  

In [3]:

X_flat = X.reshape(-1, X.shape[-1])   # Shape: (31 * 60000, 512)
y_flat = y.reshape(-1)                # Shape: (31 * 60000,)


In [17]:
unique, counts = np.unique(y, return_counts=True)
for val, count in zip(unique, counts):
    print(f"Value: {val}, Count: {count}")


Value: 0, Count: 1785931
Value: 1, Count: 74069


In [98]:
X.shape

(1, 36, 512)

In [18]:

y_binary = np.array([
    1 if label == 'Benign' else
    2 if label == 'Fuzzers' else
    0 for label in y
])

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [4]:

def evaluate(y_true, y_pred, model_name, y_scores=None):
    print(f"\n📊 Evaluation for {model_name}")

    n_classes = len(np.unique(y_true))
    average_type = 'binary' if n_classes == 2 else 'macro'

    # Accuracy and Classification Metrics
    print("Accuracy          :", accuracy_score(y_true, y_pred))
    print("Precision         :", precision_score(y_true, y_pred, average=average_type, zero_division=0))
    print("Recall            :", recall_score(y_true, y_pred, average=average_type, zero_division=0))
    print("F1-Score          :", f1_score(y_true, y_pred, average=average_type, zero_division=0))
    print("Macro F1-Score    :", f1_score(y_true, y_pred, average='macro', zero_division=0))
    print("Micro F1-Score    :", f1_score(y_true, y_pred, average='micro', zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print("Classification Report:\n", classification_report(y_true, y_pred, zero_division=0))

    # ROC-AUC and Precision-Recall Curve
    if y_scores is not None:
        try:
            if n_classes == 2:
                auc = roc_auc_score(y_true, y_scores)
                precision, recall, _ = precision_recall_curve(y_true, y_scores)
                plt.plot(recall, precision, label=f"{model_name} (AUC = {auc:.2f})")
                print("ROC-AUC Score     :", auc)
            # For Multiclass Classification (n_classes > 2)
            else:
                # Plot PR curve for each class in a one-vs-rest fashion
                for i in range(n_classes):
                    precision, recall, _ = precision_recall_curve(y_true == i, y_scores[:, i])
                    plt.plot(recall, precision, label=f"Class {i} (PR Curve)")

                # Calculate Multiclass ROC-AUC (One-vs-Rest style)
                auc = roc_auc_score(y_true, y_scores, multi_class='ovr', average='macro')
                print("Multiclass ROC-AUC (macro, ovr):", auc)

        except Exception as e:
            print("⚠️ ROC-AUC or PR Curve could not be plotted:", str(e))

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X_flat, y_flat, test_size=0.3, random_state=42)
results = {}

# --- 1. KNN ---
knn = NearestNeighbors(n_neighbors=5)
knn.fit(X_train)
dists, _ = knn.kneighbors(X_test)
knn_scores = dists[:, -1]
knn_preds = (knn_scores > np.percentile(knn_scores, 90)).astype(int)
results["KNN"] = (knn_preds, knn_scores)

# --- 2. Isolation Forest ---
iso = IsolationForest(contamination=0.1, random_state=42)
iso.fit(X_train)
iso_scores = -iso.decision_function(X_test)
iso_preds = (iso_scores > np.percentile(iso_scores, 90)).astype(int)
results["Isolation Forest"] = (iso_preds, iso_scores)

# --- 3. LOF ---
lof = LocalOutlierFactor(n_neighbors=20, novelty=True)
lof.fit(X_train)
lof_scores = -lof.decision_function(X_test)
lof_preds = (lof_scores > np.percentile(lof_scores, 90)).astype(int)
results["LOF"] = (lof_preds, lof_scores)

# --- 4. PCA Reconstruction ---
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)
X_test_recon = pca.inverse_transform(X_test_pca)
pca_scores = np.mean((X_test - X_test_recon)**2, axis=1)
pca_preds = (pca_scores > np.percentile(pca_scores, 90)).astype(int)
results["PCA"] = (pca_preds, pca_scores)

# --- 5. One-Class SVM ---
svm = OneClassSVM(kernel="rbf", nu=0.1, gamma='scale')
svm.fit(X_train)
svm_scores = -svm.decision_function(X_test)
svm_preds = (svm_scores > np.percentile(svm_scores, 90)).astype(int)
results["One-Class SVM"] = (svm_preds, svm_scores)

# # --- 6. Autoencoder ---
# def build_autoencoder(input_dim):
#     inp = Input(shape=(input_dim,))
#     x = Dense(64, activation='relu')(inp)
#     x = Dense(32, activation='relu')(x)
#     bottleneck = Dense(16, activation='relu')(x)
#     x = Dense(32, activation='relu')(bottleneck)
#     x = Dense(64, activation='relu')(x)
#     out = Dense(input_dim, activation='linear')(x)
#     return Model(inp, out)

# auto = build_autoencoder(X.shape[1])
# auto.compile(optimizer=Adam(1e-3), loss='mse')
# auto.fit(X_train, X_train, epochs=20, batch_size=32, verbose=0, validation_split=0.1)
# X_test_recon = auto.predict(X_test)
# ae_scores = np.mean((X_test - X_test_recon)**2, axis=1)
# ae_preds = (ae_scores > np.percentile(ae_scores, 90)).astype(int)
# results["Autoencoder"] = (ae_preds, ae_scores)


print(len(np.unique(y_test)))
# ------- Evaluate All Models -------
plt.figure(figsize=(10, 6))
for model_name, (preds, scores) in results.items():
    
    evaluate(y_test, preds, model_name, scores)

# --- Show PR Curve ---
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.grid(True)
plt.show()

2025-05-18 14:57:15.999154: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Max
2025-05-18 14:57:15.999187: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2025-05-18 14:57:15.999199: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
2025-05-18 14:57:15.999225: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-05-18 14:57:15.999241: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


ValueError: Input 0 of layer "functional" is incompatible with the layer: expected shape=(None, 60000), found shape=(None, 512)